# Código atualizado


In [ ]:
# ============================================
# 0) Imports
# ============================================
import warnings

import numpy as np
import pandas as pd

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier 

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
RNG_SEED = 42

OUTPUT_DIR = 'output'

DATASET_K8 = OUTPUT_DIR + '/dataset_k8.csv'
DATASET_K11 = OUTPUT_DIR + '/dataset_k11.csv'
DATASET_K16 = OUTPUT_DIR + '/dataset_k16.csv'
DATASET_K8_TSALLIS = OUTPUT_DIR + '/dataset_k8_tsallis.csv'
DATASET_K11_TSALLIS = OUTPUT_DIR + '/dataset_k11_tsallis.csv'
DATASET_K16_TSALLIS = OUTPUT_DIR + '/dataset_k16_tsallis.csv'

RESULT_K8 = OUTPUT_DIR + '/resultados_k8.csv'
RESULT_K11 = OUTPUT_DIR + '/resultados_k11.csv'
RESULT_K16 = OUTPUT_DIR + '/resultados_k16.csv'
RESULT_K8_TSALLIS = OUTPUT_DIR + '/resultados_k8_tsallis.csv'
RESULT_K11_TSALLIS = OUTPUT_DIR + '/resultados_k11_tsallis.csv'
RESULT_K16_TSALLIS = OUTPUT_DIR + '/resultados_k16_tsallis.csv'

np.random.seed(RNG_SEED)


In [ ]:
# ============================================
# 1) Carregar datasets
# ============================================
df_k8 = pd.read_csv(DATASET_K8)
df_k11 = pd.read_csv(DATASET_K11)
df_k16 = pd.read_csv(DATASET_K16)
df_k8_tsallis = pd.read_csv(DATASET_K8_TSALLIS)
df_k11_tsallis = pd.read_csv(DATASET_K11_TSALLIS)
df_k16_tsallis = pd.read_csv(DATASET_K16_TSALLIS)

print("Shape k8: ", df_k8.shape)
print("Shape k11: ", df_k11.shape)
print("Shape k16: ", df_k16.shape)
print("Shape k8_tsallis: ", df_k8_tsallis.shape)
print("Shape k11_tsallis: ", df_k11_tsallis.shape)
print("Shape k16_tsallis: ", df_k16_tsallis.shape)

Shape k8:  (81, 10)
Shape k11:  (81, 13)
Shape k16:  (81, 18)


In [ ]:
# ============================================
# 2) Preparar X, y e grupos
# Removemos metadados e o target das features
# ============================================

cols_to_drop = ["status", "subject_id"]
feature_cols_k8 = [c for c in df_k8.columns if c not in cols_to_drop]
feature_cols_k11 = [c for c in df_k11.columns if c not in cols_to_drop]
feature_cols_k16 = [c for c in df_k16.columns if c not in cols_to_drop]
feature_cols_k8_tsallis = [c for c in df_k8_tsallis.columns if c not in cols_to_drop]
feature_cols_k11_tsallis = [c for c in df_k11_tsallis.columns if c not in cols_to_drop]
feature_cols_k16_tsallis = [c for c in df_k16_tsallis.columns if c not in cols_to_drop]


X_k8 = df_k8[feature_cols_k8]
X_k11 = df_k11[feature_cols_k11]
X_k16 = df_k16[feature_cols_k16]
X_k8_tsallis = df_k8_tsallis[feature_cols_k8_tsallis]
X_k11_tsallis = df_k11_tsallis[feature_cols_k11_tsallis]
X_k16_tsallis = df_k16_tsallis[feature_cols_k16_tsallis]

y = df_k16["status"].astype(int)
groups = df_k16["subject_id"]

print("Colunas a serem descartadas:", cols_to_drop)
print("Colunas de features k8:", feature_cols_k8)
print(len(feature_cols_k8), "features selecionadas.")
print("Colunas de features k11:", feature_cols_k11)
print(len(feature_cols_k11), "features selecionadas.")
print("Colunas de features k16:", feature_cols_k16)
print(len(feature_cols_k16), "features selecionadas.")
print("Colunas de features k8_tsallis:", feature_cols_k8_tsallis)
print(len(feature_cols_k8_tsallis), "features selecionadas.")
print("Colunas de features k11_tsallis:", feature_cols_k11_tsallis)
print(len(feature_cols_k11_tsallis), "features selecionadas.")
print("Colunas de features k16_tsallis:", feature_cols_k16_tsallis)
print(len(feature_cols_k16_tsallis), "features selecionadas.")

print("\nBalanceamento:")
print(pd.Series(y).value_counts().rename(index={0: "Controle(0)", 1: "Parkinson(1)"}))

Colunas a serem descartadas: ['status', 'subject_id']
Colunas de features k8: ['dmfcc6_std', 'mfcc13_std', 'mfcc10_std', 'mfcc11_std', 'dmfcc11_std', 'mfcc2_mean', 'spec_centroid_mean_hz', 'mfcc9_std']
8 features selecionadas.
Colunas de features k11: ['dmfcc6_std', 'mfcc13_std', 'mfcc10_std', 'mfcc11_std', 'dmfcc11_std', 'mfcc2_mean', 'spec_centroid_mean_hz', 'mfcc9_std', 'mfcc8_std', 'dmfcc3_mean', 'mfcc3_std']
11 features selecionadas.
Colunas de features k16: ['dmfcc6_std', 'mfcc13_std', 'mfcc10_std', 'mfcc11_std', 'dmfcc11_std', 'mfcc2_mean', 'spec_centroid_mean_hz', 'mfcc9_std', 'mfcc8_std', 'dmfcc3_mean', 'mfcc3_std', 'shimmer_apq3', 'dmfcc13_mean', 'dmfcc4_std', 'jitter_ppq5', 'mfcc1_std']
16 features selecionadas.

Balanceamento:
status
Controle(0)     41
Parkinson(1)    40
Name: count, dtype: int64


In [5]:
def make_pipeline(model):
    return ImbPipeline(steps=[  
        # 1. RobustScaler: Escalonamento robusto baseado em quartis
        ("scale", RobustScaler()),

        # 2. SMOTE: Balanceamento sintético aplicado apenas durante o 'fit' (treino)
        ("smote", SMOTE(random_state=RNG_SEED, k_neighbors=3)),

        # 3. O Classificador (SVC, XGBoost, etc.)
        ("clf", model)
    ])
    

In [6]:
# =========================================================
# 13) Benchmark: roda GroupKFold CV e devolve tabela resumo
# =========================================================
def benchmark_models(X, y, groups, models_dict, n_splits=5, n_jobs=1):
    cv = GroupKFold(n_splits=n_splits)

    scoring = {
        "bal_acc": "balanced_accuracy",
        "roc_auc": "roc_auc",
        "f1": "f1",
    }

    rows = []
    for name, model in models_dict.items():
        pipe = make_pipeline(model)

        scores = cross_validate(
            pipe, X, y,
            groups=groups,
            cv=cv,
            scoring=scoring,
            n_jobs=n_jobs,
            error_score="raise"
        )

        rows.append({
            "model": name,
            "bal_acc_mean": float(np.mean(scores["test_bal_acc"])),
            "bal_acc_std":  float(np.std(scores["test_bal_acc"])),
            "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
            "roc_auc_std":  float(np.std(scores["test_roc_auc"])),
            "f1_mean":      float(np.mean(scores["test_f1"])),
            "f1_std":       float(np.std(scores["test_f1"])),
        })

    return pd.DataFrame(rows).sort_values("bal_acc_mean", ascending=False).reset_index(drop=True)


In [ ]:
# =========================================================
# 15) Modelos Classicos - Ajuste fino
# =========================================================

models = {
    "logreg_l2": LogisticRegression(
            C=0.5,
            solver="liblinear",
            max_iter=5000,
            class_weight="balanced",
            random_state=RNG_SEED,
        ),

    "svc_rbf":  SVC(
            C=1.0,
            gamma="scale",
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=RNG_SEED,
        ),
    
    "linear_svc_cal": CalibratedClassifierCV(
            estimator=LinearSVC(
                C=0.1,
                class_weight="balanced",
                random_state=RNG_SEED,
                max_iter=10000,
                dual=False
            ),
            method="sigmoid",
            cv=3,
        ),
    
    "random_forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=3,
            min_samples_split=10,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RNG_SEED,
            n_jobs=-1,
        ),
    
    "gradient_boosting": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.02,
            max_depth=2,
            subsample=0.6,
            random_state=RNG_SEED,
        ),
    
    "adaboost": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=1, 
                random_state=RNG_SEED
                ),
            n_estimators=100,
            learning_rate=0.5,            
            random_state=RNG_SEED,
        ),
    
    "decision_tree": DecisionTreeClassifier(
            max_depth=2,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RNG_SEED,
        ),
    
    "knn": KNeighborsClassifier(
            n_neighbors=7,
            weights="uniform",
            metric="minkowski",
            p=2,
        ),
    
    "mlp": MLPClassifier(
            hidden_layer_sizes=(4,),
            solver='lbfgs',
            alpha=1.0,
            max_iter=500,
            random_state=RNG_SEED,
            early_stopping=True,
            validation_fraction=0.1
        ),
    
    "xgboost": XGBClassifier(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.02,
            subsample=0.6,          
            colsample_bytree=0.6,   
            gamma=2,                
            reg_alpha=1.0,          
            reg_lambda=2.0,         
            min_child_weight=5,
            objective="binary:logistic",
            random_state=RNG_SEED,
            n_jobs=-1,
            verbosity=0,
        ),
    
    "catboost": CatBoostClassifier(
        iterations=200, 
        depth=2, 
        learning_rate=0.02,
        l2_leaf_reg=10,          
        bootstrap_type='Bernoulli',
        subsample=0.5,
        loss_function='Logloss', 
        verbose=0, 
        random_seed=RNG_SEED,
        allow_writing_files=False 
    ),

    "lightgbm": LGBMClassifier(
        n_estimators=100, 
        learning_rate=0.02, 
        num_leaves=3,
        min_child_samples=5,
        boosting_type='goss',
        random_state=RNG_SEED, 
        n_jobs=-1, 
        colsample_bytree=0.5,
        importance_type='gain',
        verbosity=-1,
    )
}


In [8]:
# =========================================================
# 16) Rodar benchmark (GroupKFold por sujeito)
# =========================================================
results_k8 = benchmark_models(X_k8, y, groups, models, n_splits=5, n_jobs=1)
results_k11 = benchmark_models(X_k11, y, groups, models, n_splits=5, n_jobs=1)
results_k16 = benchmark_models(X_k16, y, groups, models, n_splits=5, n_jobs=1)

# Mostrar ranking
print("\n=== Ranking por Balanced Accuracy (GroupKFold, sem leakage) ===")
print("\n=== Ranking k8 ===")
print(results_k8)
print("\n=== Ranking k11 ===")
print(results_k11)
print("\n=== Ranking k16 ===")
print(results_k16)


=== Ranking por Balanced Accuracy (GroupKFold, sem leakage) ===

=== Ranking k8 ===
                model  bal_acc_mean  bal_acc_std  roc_auc_mean  roc_auc_std  \
0             svc_rbf      0.776389     0.115654      0.765625     0.088388   
1             xgboost      0.763889     0.073886      0.830208     0.048255   
2            catboost      0.740278     0.090502      0.780556     0.055039   
3            lightgbm      0.740278     0.127868      0.777083     0.038330   
4   gradient_boosting      0.738889     0.127642      0.770833     0.075332   
5       random_forest      0.727778     0.108155      0.805208     0.046187   
6      linear_svc_cal      0.713889     0.115654      0.796181     0.091238   
7           logreg_l2      0.701389     0.107403      0.793403     0.088722   
8                 knn      0.691667     0.112114      0.768403     0.100013   
9       decision_tree      0.690278     0.039917      0.713889     0.053897   
10                mlp      0.688889     0.1425

In [9]:
# -------------- salvando resultados ------------------
results_k8.to_csv(RESULT_K8, index=False)
print(f'Resultados salvos no arquivo {RESULT_K8}')

results_k8.to_csv(RESULT_K11, index=False)
print(f'Resultados salvos no arquivo {RESULT_K11}')

results_k8.to_csv(RESULT_K16, index=False)
print(f'Resultados salvos no arquivo {RESULT_K16}')

Resultados salvos no arquivo output/resultados_k8.csv
Resultados salvos no arquivo output/resultados_k11.csv
Resultados salvos no arquivo output/resultados_k16.csv
